# 实践 8.1  G×E反应规范与单环境留出

《统计遗传与智能育种学》第八章 · 电子资源 `08_enviromics_gxe.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：08_enviromics_gxe.ipynb。实践模拟30个基因型在6个环境中的表现，每个环境包含温度与水分协变量；不同基因型具有不同环境敏感度。代码输出各环境汇总与基因型特异反应规范参数，并比较共享基因型/环境信息的随机记录划分与整组留出第5号环境的评价。最后利用环境协变量预测一个新环境，并要求读者判断该预测属于插值还是外推。该模型没有基因组关系信息，因此不声称能预测训练中从未出现的新基因型。

运行检查：重启内核并运行全部单元  
预期：输出环境协变量表、反应规范截距/斜率；固定种子下随机记录留出相关约0.917，留出第5号环境时相关约0.857、均方根误差（root mean squared error, RMSE）约0.768；解释为何二者对应不同信息场景，并说明当前模型不能预测全新基因型。

## 输入、完整代码、实际输出与解释

输入与可复现条件：30个基因型×6个环境，环境含温度和水分协变量，不同基因型具有不同环境敏感度；SEED=801。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=801
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 801


In [2]:
ng,ne=30,6; temp=np.linspace(-1.5,1.5,ne); water=np.array([.9,.3,.7,-.2,.1,-.7]); env_index=.7*temp-.5*water; base=rng.normal(0,1,ng); sens=rng.normal(.5,.45,ng); rows=[]
for g in range(ng):
 for e in range(ne): rows.append((g,e,temp[e],water[e],base[g]+sens[g]*env_index[e]+rng.normal(0,.45)))
df=pd.DataFrame(rows,columns=['genotype','env','temp','water','y']); print(df.groupby('env')[['temp','water','y']].mean().round(3))

# 基因型特异反应规范。它只能预测训练中出现过的基因型；这里专门评价环境外推。
from sklearn.metrics import mean_squared_error
def reaction_predict(train,test):
 pred=[]
 for _,r in test.iterrows():
  sub=train[train.genotype==r.genotype]; X=np.column_stack([np.ones(len(sub)),.7*sub.temp-.5*sub.water]); b=np.linalg.lstsq(X,sub.y,rcond=None)[0]; pred.append(b[0]+b[1]*(.7*r.temp-.5*r.water))
 return np.array(pred)
coef=[]
for g in range(ng):
 sub=df[df.genotype==g]; X=np.column_stack([np.ones(len(sub)),.7*sub.temp-.5*sub.water]); coef.append(np.linalg.lstsq(X,sub.y,rcond=None)[0])
print('reaction-norm [intercept, slope] first5\n',np.round(np.asarray(coef)[:5],3))
idx=np.arange(len(df)); rng.shuffle(idx); tr=idx[:140]; te=idx[140:]; pr=reaction_predict(df.iloc[tr],df.iloc[te]); print('shared-genotype/env random-record r',round(np.corrcoef(pr,df.y.iloc[te])[0,1],3))
hold=5; train=df[df.env!=hold]; test=df[df.env==hold]; ph=reaction_predict(train,test); print('leave-env-out r',round(np.corrcoef(ph,test.y)[0,1],3),'RMSE',round(np.sqrt(mean_squared_error(test.y,ph)),3))
# 未见环境：只要EC已知，数值上可预测；是否属于可靠插值/外推仍须检查训练环境覆盖。
new_temp,new_water=.4,.0; new_idx=.7*new_temp-.5*new_water; pred_new=[]
for g in range(ng):
 sub=df[df.genotype==g]; X=np.column_stack([np.ones(ne),env_index]); b=np.linalg.lstsq(X,sub.y,rcond=None)[0]; pred_new.append(b[0]+b[1]*new_idx)
print('new-environment predicted phenotype first5',np.round(pred_new[:5],3))


     temp  water      y
env                    
0    -1.5    0.9 -0.840
1    -0.9    0.3 -0.478
2    -0.3    0.7 -0.183
3     0.3   -0.2  0.053
4     0.9    0.1  0.247
5     1.5   -0.7  0.804
reaction-norm [intercept, slope] first5
 [[-0.715  0.722]
 [-0.555  0.424]
 [ 1.808  0.652]
 [ 1.126  0.18 ]
 [-0.381  0.872]]
shared-genotype/env random-record r 0.917
leave-env-out r 0.857 RMSE 0.768
new-environment predicted phenotype first5 [-0.513 -0.436  1.991  1.176 -0.136]


结果解释：随机记录划分中训练和测试共享基因型/环境信息，相关0.917；留出第5号环境时相关0.857、均方根误差（RMSE）为0.768，任务更接近预测未见环境。新环境预测只对已见基因型成立；当前模型没有基因组关系信息，不能声称预测全新基因型，超出训练环境范围还涉及外推风险。

可修改练习：分别留出中间环境和边界环境，比较预测误差。另调高新环境temp，按合成环境指数判断插值或外推。